# Test PSTH and PCA methods of Session class with synthetic sata

This notebook tests the Session class methods using controlled synthetic data to verify functionality.

In [ ]:
import pandas as pd
import numpy as np
import holoviews as hv
import hvplot.pandas
import panel as pn

from holoviews import opts
from bokeh.io import output_notebook

# Import the session analysis class with reload capability
import importlib
import session_class
importlib.reload(session_class)
from session_class import Session

print("Session class imported successfully!")

output_notebook()
hv.extension('bokeh')

Session class imported successfully!


Loading BokehJS ...

## Test 1: 20-Cell Session with Direction-Selective Activity

Create a session with 20 cells, each with 2 GO trials (left and right):
- **GO Left (dir=180)**: Cell i spikes at 90+i ms (before go_cue)
- **GO Right (dir=0)**: Cell i spikes at 110-i ms (after go_cue)
- **go_cue**: 100ms for all trials

This creates a gradient pattern where:
- Left: Earlier cells spike first (cell 0 at 90ms, cell 19 at 109ms)
- Right: Later cells spike first (cell 19 at 91ms, cell 0 at 110ms)

In [62]:
def create_gradient_session_data(n_cells=20):
    """
    Create session with direction-selective gradient activity.
    
    Parameters:
    -----------
    n_cells : int
        Number of cells (default: 20)
    
    Returns:
    --------
    pd.DataFrame with trial data for all cells
    """
    test_session = 'test_session_gradient'
    go_cue_time = 100
    
    trials = []
    
    for cell_id in range(n_cells):
        # GO Left trial: cell i spikes at 90+i
        trials.append({
            'cell_ID': cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 180,  # Left
            'trial_failed': False,
            'go_cue': go_cue_time,
            'stop_cue': np.nan,
            'first_relevant_saccade': 250,
            'ssd_number': np.nan,
            'neural_data': [90 + cell_id],  # Spike at 90+i
            'trial_number': cell_id * 2 + 1,
        })
        
        # GO Right trial: cell i spikes at 110-i
        trials.append({
            'cell_ID': cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 0,  # Right
            'trial_failed': False,
            'go_cue': go_cue_time,
            'stop_cue': np.nan,
            'first_relevant_saccade': 250,
            'ssd_number': np.nan,
            'neural_data': [110 - cell_id],  # Spike at 110-i
            'trial_number': cell_id * 2 + 2,
        })
    
    return pd.DataFrame(trials)

# Create session
gradient_df = create_gradient_session_data(n_cells=20)
gradient_session = Session(gradient_df, verbose=True)

print(f"\nCreated session with {gradient_session.n_cells} cells and {gradient_session.n_trials} trials")

Session test_session_gradient initialized:
  - Number of cells: 20
  - Total trials: 40
  - Trial types: ['GO']
  - Directions: [np.int64(0), np.int64(180)]

Created session with 20 cells and 40 trials


### Compare Left vs Right Side-by-Side

Use the plotting method to create both heatmaps with matching cell order.

In [63]:
# Recreate the session with the updated class
gradient_session = Session(gradient_df, verbose=False)

# Create side-by-side comparison
left_plot = gradient_session.plot_population_spike_counts_heatmap(
    epok=[-20, 30],
    bin_size=1,
    alignment_point='go_cue',
    trial_type='GO',
    direction=180,
    normalize=False,
    sort_by_peak=False
) #.opts(width=400, title='Left (180°) - Gradient 90+i')

right_plot = gradient_session.plot_population_spike_counts_heatmap(
    epok=[-20, 30],
    bin_size=1,
    alignment_point='go_cue',
    trial_type='GO',
    direction=0,
    normalize=False,
    sort_by_peak=False
) #.opts(width=400, title='Right (0°) - Gradient 110-i')

# (left_plot + right_plot).cols(2)
left_plot, right_plot

(:HeatMap   [columns,index]   (value), :HeatMap   [columns,index]   (value))

In [64]:
from matplotlib import pyplot as plt


data = gradient_session.get_population_PSTH_single_condition(
    epok=[-20, 30], bin_size=1, 
    alignment_point='go_cue', trial_type='GO',
    direction=0, success_only=True, smooth=True, 
    delta=True, smooth_ker_size=5, normalize_bins=False,
    normalize=False, sort_by_peak=True
) 

print(data['sort_idx'] == np.array(data['cell_ids'], dtype=int))

heatmap = gradient_session.plot_population_PSTH_heatmap(data)

[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True]
sort_idx: [19 18 17 16 15 14 13 12 11 10  9  8  7  6  5  4  3  2  1  0]
cell_ids: [np.int64(19), np.int64(18), np.int64(17), np.int64(16), np.int64(15), np.int64(14), np.int64(13), np.int64(12), np.int64(11), np.int64(10), np.int64(9), np.int64(8), np.int64(7), np.int64(6), np.int64(5), np.int64(4), np.int64(3), np.int64(2), np.int64(1), np.int64(0)]


In [65]:
a = gradient_session.get_population_PSTH_single_condition(
    epok=[-20, 30], bin_size=1, 
    alignment_point='go_cue', trial_type='GO',
    direction=0, success_only=True, smooth=True, 
    delta=True, smooth_ker_size=5, normalize_bins=False,
    normalize=False, sort_by_peak=False
)['psth_matrix'] #['psth_matrix']

b = gradient_session.get_population_PSTH_single_condition(
    epok=[-20, 30], bin_size=1, 
    alignment_point='go_cue', trial_type='GO',
    direction=0, success_only=True, smooth=True, 
    delta=True, smooth_ker_size=5, normalize_bins=False,
    normalize=False, sort_by_peak=True
)['psth_matrix'] #['psth_matrix']

# b = b[np.arange(len(b))]
# sort_idx = data['sort_idx']
# b = b[sort_idx]
np.array_equal(a, b) 
# a, b

False

In [66]:
df = pd.DataFrame(b, index=np.arange(b.shape[0], 0, -1))
df.hvplot.heatmap()

:HeatMap   [columns,index]   (value)

In [67]:
df

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
20,-8.425711,-6.484671,-2.626181,3.071462,10.423531,19.085223,28.518472,37.999027,46.674361,53.656983,...,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000
19,-12.804211,-11.321643,-8.314968,-3.732498,2.431511,10.069080,18.897161,28.422853,37.940635,46.647594,...,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000
18,-15.700143,-14.634509,-12.427960,-8.954920,-4.086950,2.243449,9.973462,18.838769,28.396086,37.940635,...,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000
17,-17.530441,-16.806460,-15.274461,-12.782412,-9.142982,-4.182568,2.185057,9.946694,18.838769,28.396086,...,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000
16,-18.636758,-18.170392,-17.160912,-15.462523,-12.878030,-9.201374,-4.209335,2.185057,9.946694,18.838769,...,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000
15,-19.276710,-18.991209,-18.358454,-17.256530,-15.520914,-12.904797,-9.201374,-4.209335,2.185057,9.946694,...,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000
14,-19.631161,-19.464772,-19.086828,-18.416846,-17.283297,-15.520914,-12.904797,-9.201374,-4.209335,2.185057,...,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000
13,-19.819223,-19.726779,-19.523163,-19.113595,-18.416846,-17.283297,-15.520914,-12.904797,-9.201374,-4.209335,...,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000
12,-19.914841,-19.877615,-19.753546,-19.523163,-19.113595,-18.416846,-17.283297,-15.520914,-12.904797,-9.201374,...,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000
11,-19.973233,-19.941608,-19.877615,-19.753546,-19.523163,-19.113595,-18.416846,-17.283297,-15.520914,-12.904797,...,-19.973233,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000
